### Preparación del entorno

In [0]:
# Cargamos la tabla limpia y la convertimos en una Vista Temporal de Spark
df_citas = spark.table("default.citas_pmm_limpio")
df_citas.createOrReplaceTempView("v_citas_pmm")

print("Vista 'v_citas_pmm' creada exitosamente. Lista para análisis SQL.")

Vista 'v_citas_pmm' creada exitosamente. Lista para análisis SQL.


### KP1 N1: Rendimiento financiero y distribución de cobertura por aseguradora

In [0]:
%sql
SELECT 
    nom_aseguradora AS Aseguradora,
    COUNT(*) AS Volumen_Citas,
    ROUND(SUM(pago_aseg), 2) AS Total_Pagado_Aseguradora_USD,
    ROUND(SUM(pago_clie), 2) AS Total_Copago_Paciente_USD,
    ROUND(SUM(pago_total), 2) AS Ingresos_Totales_USD,
    ROUND((SUM(pago_aseg) / SUM(pago_total)) * 100, 2) AS Porcentaje_Cobertura_Promedio
FROM v_citas_pmm
WHERE estado_cita = 'Completada'
GROUP BY nom_aseguradora
ORDER BY Ingresos_Totales_USD DESC;

Aseguradora,Volumen_Citas,Total_Pagado_Aseguradora_USD,Total_Copago_Paciente_USD,Ingresos_Totales_USD,Porcentaje_Cobertura_Promedio
Particular / Sin Seguro,2067,0.0,102835.0,102835.0,0.0
PALIG,1274,44168.35,18746.65,62915.0,70.2
MAPFRE Panamá,1240,42529.43,18630.57,61160.0,69.54
Seguros ASSA,1228,42803.86,18341.14,61145.0,70.0
Compañía Internacional de Seguros (IS),1144,39688.73,16941.27,56630.0,70.08
Seguros SURA,1092,38466.83,16343.17,54810.0,70.18


### KPI N2: Tasa de ausentismo por sucursal

In [0]:
%sql
SELECT 
    nom_sucursal AS Sucursal,
    COUNT(*) AS Citas_Agendadas,
    SUM(CASE WHEN estado_cita = 'Cancelada' THEN 1 ELSE 0 END) AS Citas_Canceladas,
    SUM(CASE WHEN estado_cita = 'Completada' THEN 1 ELSE 0 END) AS Citas_Completadas,
    ROUND((SUM(CASE WHEN estado_cita = 'Cancelada' THEN 1 ELSE 0 END) / COUNT(*)) * 100, 2) AS Tasa_Cancelacion_PCT
FROM v_citas_pmm
GROUP BY nom_sucursal
ORDER BY Tasa_Cancelacion_PCT DESC;

Sucursal,Citas_Agendadas,Citas_Canceladas,Citas_Completadas,Tasa_Cancelacion_PCT
PMM San Francisco,2976,606,2370,20.36
PMM El Dorado,1955,385,1570,19.69
PMM Brisas del Golf,1527,299,1228,19.58
PMM Tocumen,986,188,798,19.07
PMM Costa del Este,2556,477,2079,18.66


### KPI N3: Segmentación demográfica y ticket promedio

In [0]:
%sql
SELECT 
    CASE 
        WHEN edad_pac_cita < 15 THEN '1. Pediátrico (<15 años)'
        WHEN edad_pac_cita BETWEEN 15 AND 64 THEN '2. Adulto (15-64 años)'
        ELSE '3. Geriátrico (65+ años)' 
    END AS Segmento_Poblacional,
    COUNT(*) AS Total_Atenciones,
    ROUND(SUM(pago_total), 2) AS Facturacion_Total_USD,
    ROUND(AVG(pago_total), 2) AS Ticket_Promedio_USD
FROM v_citas_pmm
WHERE estado_cita = 'Completada'
GROUP BY 
    CASE 
        WHEN edad_pac_cita < 15 THEN '1. Pediátrico (<15 años)'
        WHEN edad_pac_cita BETWEEN 15 AND 64 THEN '2. Adulto (15-64 años)'
        ELSE '3. Geriátrico (65+ años)' 
    END
ORDER BY Segmento_Poblacional ASC;

Segmento_Poblacional,Total_Atenciones,Facturacion_Total_USD,Ticket_Promedio_USD
1. Pediátrico (<15 años),1641,69525.0,42.37
2. Adulto (15-64 años),4666,239095.0,51.24
3. Geriátrico (65+ años),1738,90875.0,52.29


### KPI N4: Rentabilidad de cada especialidad por hora

In [0]:
%sql
SELECT 
    especialidad_medica AS Especialidad,
    COUNT(*) AS Total_Consultas,
    ROUND(SUM(mins_cit) / 60, 2) AS Horas_Totales_Invertidas,
    ROUND(SUM(pago_total), 2) AS Ingresos_Totales_USD,
    ROUND(SUM(pago_total) / (SUM(mins_cit) / 60), 2) AS Rentabilidad_Por_Hora_USD
FROM v_citas_pmm
WHERE estado_cita = 'Completada'
GROUP BY especialidad_medica
ORDER BY Rentabilidad_Por_Hora_USD DESC

Especialidad,Total_Consultas,Horas_Totales_Invertidas,Ingresos_Totales_USD,Rentabilidad_Por_Hora_USD
Neurología,447,278.5,33525.0,120.38
Nefrología,436,271.0,30520.0,112.62
Cardiología,356,221.75,24920.0,112.38
Gastroenterología,462,289.25,30030.0,103.82
Psiquiatría,467,287.25,28020.0,97.55
Radiología,674,410.5,37070.0,90.3
Otorrinolaringología,700,423.0,35000.0,82.74
Dermatología,717,448.75,35850.0,79.89
Geriatría,142,89.5,7100.0,79.33
Urología,272,166.0,12240.0,73.73


### KPI N5: Análisis de citas por jornada y hora

In [0]:
%sql
SELECT 
    CASE 
        -- REGEXP_EXTRACT busca la primera coincidencia de hora (ej: "09:30") y extrae solo los primeros 2 dígitos (\d{2})
        WHEN CAST(REGEXP_EXTRACT(hr_inicio_cit, '(\\d{2}):\\d{2}', 1) AS INT) < 12 THEN 'Mañana (07:00 - 11:59)'
        WHEN CAST(REGEXP_EXTRACT(hr_inicio_cit, '(\\d{2}):\\d{2}', 1) AS INT) BETWEEN 12 AND 16 THEN 'Tarde (12:00 - 16:59)'
        ELSE 'Noche (17:00 - 20:00)'
    END AS Jornada_Horaria,
    COUNT(*) AS Cantidad_Citas,
    ROUND((COUNT(*) / (SELECT COUNT(*) FROM v_citas_pmm)) * 100, 2) AS Porcentaje_Demanda_PCT
FROM v_citas_pmm
GROUP BY 
    CASE 
        WHEN CAST(REGEXP_EXTRACT(hr_inicio_cit, '(\\d{2}):\\d{2}', 1) AS INT) < 12 THEN 'Mañana (07:00 - 11:59)'
        WHEN CAST(REGEXP_EXTRACT(hr_inicio_cit, '(\\d{2}):\\d{2}', 1) AS INT) BETWEEN 12 AND 16 THEN 'Tarde (12:00 - 16:59)'
        ELSE 'Noche (17:00 - 20:00)'
    END
ORDER BY Cantidad_Citas DESC;

Jornada_Horaria,Cantidad_Citas,Porcentaje_Demanda_PCT
Tarde (12:00 - 16:59),5038,50.38
Mañana (07:00 - 11:59),4010,40.1
Noche (17:00 - 20:00),952,9.52


### KPI N6: Reembolso promedio de aseguradoras por especialidad

In [0]:
%sql
SELECT 
    especialidad_medica AS Especialidad,
    COUNT(*) AS Citas_Aseguradas,
    ROUND(SUM(pago_aseg), 2) AS Facturacion_A_Seguros_USD,
    ROUND(AVG(pago_aseg), 2) AS Reembolso_Promedio_Aseguradora_USD
FROM v_citas_pmm
WHERE estado_cita = 'Completada' AND nom_aseguradora != 'Sin Seguro'
GROUP BY especialidad_medica
ORDER BY Reembolso_Promedio_Aseguradora_USD DESC;

Especialidad,Citas_Aseguradas,Facturacion_A_Seguros_USD,Reembolso_Promedio_Aseguradora_USD
Neurología,447,17376.6,38.87
Nefrología,436,16215.76,37.19
Cardiología,356,12260.35,34.44
Gastroenterología,462,15642.29,33.86
Psiquiatría,467,14421.28,30.88
Radiología,674,19232.7,28.54
Geriatría,142,3781.67,26.63
Otorrinolaringología,700,18572.75,26.53
Dermatología,717,18819.8,26.25
Ginecología,262,6191.66,23.63
